# Land Classification with CNN + ViT

This is a simpler version of the full lab.

We will download the dataset, load the trained models, and compare the results from Keras and PyTorch.

## 1. Install the required libraries

This notebook uses TensorFlow, PyTorch, scikit-learn, and matplotlib.

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib torch torchvision

## 2. Import the needed libraries

We import the packages used for data handling, deep learning, and model evaluation.

In [ ]:
import os
import random
import tarfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print('Libraries ready')

## 3. Set the random seed

This helps keep the results stable from one run to another.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)
print('Seed set')

## 4. Define file paths

We define where the dataset and trained model files will be stored.

In [ ]:
data_dir = '.'
dataset_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4Z1fwRR295-1O3PMQBH6Dg/images-dataSAT.tar'
keras_model_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7uNMQhNyTA8qSSDGn5Cc7A/keras-cnn-vit-ai-capstone.keras'
pytorch_model_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/rFBrDlu1NNcAzir5Uww8eg/pytorch-cnn-vit-ai-capstone-model-state-dict.pth'

dataset_path = os.path.join(data_dir, 'images_dataSAT')
keras_model_path = os.path.join(data_dir, 'keras_cnn_vit_ai_capstone.keras')
pytorch_model_path = os.path.join(data_dir, 'pytorch_cnn_vit_ai_capstone_model_state_dict.pth')
print(dataset_path)
print(keras_model_path)
print(pytorch_model_path)

## 5. Download the dataset and model files

If the files are not already there, we download them.

In [ ]:
def download_file(url, save_path):
    if not os.path.exists(save_path):
        print(f'Downloading from {url}')
        urllib.request.urlretrieve(url, save_path)
        print('Download complete')
    else:
        print(f'File already exists: {save_path}')

download_file(dataset_url, 'images-dataSAT.tar')
if not os.path.exists(dataset_path):
    with tarfile.open('images-dataSAT.tar', 'r:*') as tar:
        tar.extractall(data_dir)
        print('Dataset extracted')

download_file(keras_model_url, keras_model_path)
download_file(pytorch_model_url, pytorch_model_path)

## 6. Prepare the dataset for evaluation

We resize images and normalize them before feeding them to the model.

In [ ]:
img_w, img_h = 64, 64
batch_size = 64
num_classes = 2
class_labels = ['non-agri', 'agri']

train_transform = transforms.Compose([
    transforms.Resize((img_w, img_h)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(dataset_path, transform=train_transform)
eval_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)
print(f'Total images: {len(full_dataset)}')

## 7. Load the Keras model

The Keras model is already trained. We just load it and use it for inference.

In [ ]:
keras_model = load_model(keras_model_path, compile=False)
keras_model.summary()

## 8. Run Keras inference

Here we collect predictions and probabilities from the Keras model.

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)
keras_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_w, img_h),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

keras_probs = keras_model.predict(keras_generator, verbose=1)
keras_preds = np.argmax(keras_probs, axis=1)
keras_labels = keras_generator.classes
print('Keras predictions ready')

## 9. Load the PyTorch model

The PyTorch model is also already trained. We load its saved state dictionary.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CNN_ViT_Hybrid(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.head(x)

pytorch_model = CNN_ViT_Hybrid(num_classes=num_classes).to(device)
pytorch_model.load_state_dict(torch.load(pytorch_model_path, map_location=device), strict=False)
pytorch_model.eval()
print('PyTorch model loaded on', device)

## 10. Run PyTorch inference

Now we get predictions and probabilities from the PyTorch model.

In [ ]:
pytorch_preds = []
pytorch_labels = []
pytorch_probs = []

with torch.no_grad():
    for images, labels in eval_loader:
        images = images.to(device)
        outputs = pytorch_model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)
        pytorch_probs.extend(probs.cpu().numpy())
        pytorch_preds.extend(preds.cpu().numpy())
        pytorch_labels.extend(labels.numpy())

print('PyTorch predictions ready')

## 11. Compare model performance

We calculate a few important metrics for both models.

In [ ]:
def summarize_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

keras_metrics = summarize_metrics(keras_labels, keras_preds, keras_probs[:, 1])
pytorch_metrics = summarize_metrics(np.array(pytorch_labels), np.array(pytorch_preds), np.array(pytorch_probs))

print('Keras metrics:', keras_metrics)
print('PyTorch metrics:', pytorch_metrics)

## 12. Final notes

This notebook shows the full workflow in a simple way: download data, load the trained models, run inference, and compare results.